# Customizing the OTC Cycle

The observe-think-act (OTC) cycle is the heartbeat of every amphibious agent. At each stage, the framework provides hook methods you can override to inject custom behavior — whether it's enriching observations, reshaping LLM messages, intercepting tool calls, or post-processing outputs.

These hooks exist at two levels: **Worker level** (per-worker customization) and **Agent level** (shared across all workers). When a worker hook returns the special `_DELEGATE` sentinel, the framework falls through to the agent-level hook.

In this tutorial, we'll build a **security audit agent** that uses OTC hooks to inject system state, enforce safety rules, filter dangerous operations, and sanitize output.

## Initialize

First, let's set up the LLM and define the security-themed tools our audit agent will use.

In [1]:
import os

model_name = os.environ.get("MODEL_NAME")
api_key = os.environ.get("API_KEY")
api_base = os.environ.get("BASE_URL")

In [ ]:
from bridgic.llms.openai import OpenAILlm, OpenAIConfiguration

llm = OpenAILlm(
    api_key=api_key,
    api_base=api_base,
    timeout=120,
    configuration=OpenAIConfiguration(
        model=model_name,
        temperature=0.0,
        max_tokens=16384,
    ),
)

In [3]:
from bridgic.core.agentic.tool_specs import FunctionToolSpec


async def list_files(directory: str) -> str:
    """List files in a directory"""
    return f"Files in {directory}: config.yaml, app.log, secrets.env, data.db"


async def read_file(filepath: str) -> str:
    """Read the content of a file"""
    if "secret" in filepath.lower() or ".env" in filepath:
        return f"[SENSITIVE] Content of {filepath}: API_KEY=sk-xxx, DB_PASS=yyy"
    return f"Content of {filepath}: (normal file content)"


async def delete_file(filepath: str) -> str:
    """Delete a file from the system"""
    return f"Deleted {filepath}"


async def check_security_status() -> str:
    """Check system security status"""
    return "System status: 2 warnings, 0 critical, last scan: 1 hour ago"


list_files_tool = FunctionToolSpec.from_raw(list_files)
read_file_tool = FunctionToolSpec.from_raw(read_file)
delete_file_tool = FunctionToolSpec.from_raw(delete_file)
check_security_status_tool = FunctionToolSpec.from_raw(check_security_status)

We have four tools:

- **`list_files`** — lists files in a given directory. Returns a mix of normal and sensitive files.
- **`read_file`** — reads a file's content. If the file is sensitive (e.g., `.env` or `secrets`), it returns data containing API keys and passwords.
- **`delete_file`** — deletes a file. This is the dangerous operation we want to block.
- **`check_security_status`** — reports the system's current security posture.

## Part 1: Observe Phase — Injecting Custom Perception

The `observation()` hook lets you inject custom context before the LLM starts thinking. It exists at two levels:

- **Worker-level `observation()`**: Gives a specific worker its own view of the environment — for example, injecting security policies or audit metadata.
- **Agent-level `observation()`**: Provides shared context for all workers — for example, system-wide information like server name and region.
- When a worker returns `_DELEGATE`, the framework falls through to the agent's `observation()` instead.

Let's build a security worker that injects audit-specific context into the observe phase.

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker,
    think_unit
)


class SecurityWorker(CognitiveWorker):
    """A worker with custom observation that injects security logs."""

    async def thinking(self) -> str:
        return (
            "Analyze the system for security issues. "
            "Check files, review the security status, and report findings. "
            "NEVER delete files or access sensitive files like .env or secrets."
        )

    async def observation(self, context: CognitiveContext):
        """Inject security-specific context before thinking."""
        return (
            f"Current goal: {context.goal}\n"
            f"Security policy: Read-only audit mode. File deletion is PROHIBITED.\n"
            f"Audit timestamp: 2024-06-15T10:30:00Z"
        )


class SecurityAuditAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(SecurityWorker(), max_attempts=5)

    async def observation(self, ctx: CognitiveContext):
        """Agent-level observation: provides system-wide context for all workers."""
        return f"System: production-server-01, Region: us-east-1, Uptime: 45 days"

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor

In [7]:
agent = SecurityAuditAgent(llm=llm, verbose=True)
result = await agent.arun(
    goal="Perform a security audit on the /var/app directory",
    tools=[list_files_tool, read_file_tool, delete_file_tool, check_security_status_tool],
)
print(result)

[17:03:11.426] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[17:03:11.427] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[17:03:11.428] [Observe] (_amphibious_automa.py:861) SecurityWorker: Current goal: Perform a security audit on the /var/app directory
Security policy: Read-only audit mode. File deletion is PROHIBITED.
Audit timestamp: 2024-06-15T10:30:00Z
[17:03:17.463] [Think] (_amphibious_automa.py:867) SecurityWorker: finish=False, step=Starting security audit on /var/app directory. I will first list the files in the directory and check the system security status to understand what we're working with. Following the read-only audit policy - no file deletions will be performed.
[17:03:17.465] [Act] (_amphibious_automa.py:873) SecurityWorker:
{
    "content": "Starting security audit on /var/app directory. I will first list the files in the directory and check the system security status to understand what we're working with. Following the read-only

Notice how the `SecurityWorker` injects its own observation — the security policy and audit timestamp — into the context before the LLM reasons about the task. Because the worker defines its own `observation()`, it does **not** fall through to the agent-level observation.

If you wanted a worker to use the agent-level observation, just do nothing.

This two-tier pattern lets you share common context across workers while allowing individual workers to override it when needed.

## Part 2: Think Phase — Reshaping LLM Messages

The `build_messages()` hook controls how the prompt is assembled before being sent to the LLM. By overriding it, you can inject mandatory rules, restructure the message format, or add custom system instructions.

Let's create a worker that injects strict security rules into every LLM call.

In [10]:
from bridgic.amphibious._cognitive_worker import Message


class StrictSecurityWorker(CognitiveWorker):
    async def thinking(self) -> str:
        return "Perform a security audit of the specified directory. Respond in JSON format."

    async def build_messages(
        self,
        think_prompt: str,
        tools_description: str,
        output_instructions: str,
        context_info: str,
    ):
        """Inject mandatory security rules into the system message."""
        security_rules = (
            "\n\n=== MANDATORY SECURITY RULES ===\n"
            "1. NEVER call delete_file under any circumstances.\n"
            "2. NEVER read files ending in .env or containing 'secret'.\n"
            "3. Always check security status before accessing any file.\n"
            "4. Report all findings without exposing sensitive data.\n"
        )

        system_content = f"{think_prompt}{security_rules}\n\n{tools_description}\n\n{output_instructions}"

        return [
            Message.from_text(text=system_content, role="system"),
            Message.from_text(text=context_info, role="user"),
        ]

The `build_messages()` method receives four components:

- **`think_prompt`** — the thinking instruction from `thinking()`.
- **`tools_description`** — auto-generated descriptions of available tools.
- **`output_instructions`** — formatting rules for the LLM's response.
- **`context_info`** — the assembled context (goal, observation, history, etc.).

By overriding this method, you have full control over the structure and content of the messages sent to the LLM. Here, we insert mandatory security rules that the LLM must follow — regardless of what the user's goal says.

Let's use this worker in an agent.

In [11]:
class StrictAuditAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(StrictSecurityWorker(verbose_prompt=True), max_attempts=5)

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor


agent = StrictAuditAgent(llm=llm, verbose=True)
result = await agent.arun(
    goal="Audit the /var/app directory for security issues",
    tools=[list_files_tool, read_file_tool, delete_file_tool, check_security_status_tool],
)
print(result)

[17:09:12.012] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[17:09:12.012] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[17:09:12.013] [Observe] (_amphibious_automa.py:861) StrictSecurityWorker: None
[17:09:18.136] [Think] (_cognitive_worker.py:332) Message 1 (system, 390 tokens):
Perform a security audit of the specified directory. Respond in JSON format.

=== MANDATORY SECURITY RULES ===
1. NEVER call delete_file under any circumstances.
2. NEVER read files ending in .env or containing 'secret'.
3. Always check security status before accessing any file.
4. Report all findings without exposing sensitive data.


# Available Tools (with parameters):
• list_files: List files in a directory
  - directory (string) [required]
• read_file: Read the content of a file
  - filepath (string) [required]
• delete_file: Delete a file from the system
  - filepath (string) [required]
• check_security_status: Check system security status

# Context Acquiring
If the 

Even though the `delete_file` tool is available, the injected security rules instruct the LLM to never call it. The `build_messages()` hook gives you a clean way to add policies at the prompt level — the LLM sees the rules as part of its instructions.

## Part 3: Act Phase — Intercepting Tool Calls

The act phase provides hooks to intercept and modify tool execution. These hooks give you programmatic control over what actually happens when the LLM decides to call a tool.

### `before_action` — Filtering Dangerous Calls

The `before_action()` hook runs after the LLM makes its decision but before any tools are executed. This is your last line of defense — you can block, modify, or filter tool calls regardless of what the LLM decided.

The `decision_result` parameter is a `List[Tuple[ToolCall, ToolSpec]]` — each element is a pair of:
- **`ToolCall`** — the LLM's request (`name`, `arguments`, `id`)
- **`ToolSpec`** — the matching tool specification (`tool_name`, `tool_description`, `tool_parameters`)

In [5]:
class SafeAuditAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(
        CognitiveWorker.inline("Audit the system and report security findings. Respond in JSON format."),
        max_attempts=5,
    )

    async def before_action(self, decision_result, ctx):
        """Filter out any dangerous tool calls before execution."""
        if isinstance(decision_result, list):
            blocked_tools = {"delete_file"}
            filtered = []
            for tool_call, tool_spec in decision_result:
                if tool_spec.tool_name in blocked_tools:
                    print(f"[BLOCKED] Prevented call to: {tool_spec.tool_name}")
                else:
                    filtered.append((tool_call, tool_spec))
            return filtered if filtered else decision_result
        return decision_result

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor

In [6]:
agent = SafeAuditAgent(llm=llm, verbose=True)
result = await agent.arun(
    goal="Audit /var/app and clean up unnecessary files",
    tools=[list_files_tool, read_file_tool, delete_file_tool, check_security_status_tool],
)
print(result)

[17:25:05.947] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[17:25:05.947] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[17:25:05.948] [Observe] (_amphibious_automa.py:861) _PromptWorker: None
[17:25:11.238] [Think] (_amphibious_automa.py:867) _PromptWorker: finish=False, step=Starting the security audit of /var/app directory. First, I need to explore what files exist in this directory and check the overall system security status to understand the current state before identifying unnecessary files to clean up.
[17:25:11.241] [Act] (_amphibious_automa.py:873) _PromptWorker:
{
    "content": "Starting the security audit of /var/app directory. First, I need to explore what files exist in this directory and check the overall system security status to understand the current state before identifying unnecessary files to clean up.",
    "result": {
        "results": [
            {
                "tool_id": "call_0",
                "tool_name": "list_fil

Even though the goal explicitly asks to "clean up unnecessary files," the `before_action` hook blocks any calls to `delete_file`. This is a programmatic safety net — unlike prompt-level rules that the LLM might occasionally ignore, `before_action` is enforced in code and cannot be bypassed.

The key difference between `before_action` and `build_messages`:

- **`build_messages`** (Think phase): Persuades the LLM not to call dangerous tools. Effective but not guaranteed.
- **`before_action`** (Act phase): Programmatically blocks dangerous tools. Guaranteed enforcement.

### `action_custom_output` — Post-Processing Structured Output

When a `CognitiveWorker` is configured with `output_schema`, the LLM produces a typed Pydantic instance instead of tool calls. In this mode, the framework calls `action_custom_output()` instead of `action_tool_call()`.

This hook lets you post-process the structured output before it's recorded in the context — for example, sanitizing sensitive data, adding metadata, or validating business rules.

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class AuditReport(BaseModel):
    """Structured output schema for the audit worker."""
    findings: List[str] = Field(description="List of security findings")
    risk_level: str = Field(description="Overall risk level: low, medium, or high")
    recommendations: List[str] = Field(description="Recommended actions")


class SanitizingAgent(AmphibiousAutoma[CognitiveContext]):
    # output_schema tells the worker to produce an AuditReport instead of tool calls
    auditor = think_unit(
        CognitiveWorker.inline(
            "Analyze the security status and produce a structured audit report. "
            "List the findings, assess risk level, and provide recommendations.",
            output_schema=AuditReport,
        ),
        max_attempts=1,
    )

    async def action_custom_output(self, decision_result, ctx):
        """Redact sensitive information from structured output."""
        if isinstance(decision_result, AuditReport):
            decision_result.findings = [
                f.replace("sk-xxx", "[REDACTED]").replace("yyy", "[REDACTED]")
                for f in decision_result.findings
            ]
            print(f"[action_custom_output] Sanitized {len(decision_result.findings)} findings")
        return decision_result

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor


agent = SanitizingAgent(llm=llm, verbose=True)
result = await agent.arun(
    goal="Produce a security audit report for /var/app. "
         "Known facts: files include config.yaml, app.log, secrets.env (contains API_KEY=sk-xxx, DB_PASS=yyy), data.db. "
         "System status: 2 warnings, 0 critical.",
)
print(result)

[17:26:49.335] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[17:26:49.336] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[17:26:49.336] [Observe] (_amphibious_automa.py:861) _PromptWorker: None
[17:27:05.063] [Think] (_amphibious_automa.py:867) _PromptWorker: finish=True, step=Analyzed the /var/app directory contents and system status. Key findings include exposed secrets in secrets.env (API_KEY and DB_PASS in plaintext), potential log file sensitivity in app.log, and possible misconfigurations in config.yaml or data.db. Although system status reports 0 critical issues, plaintext secrets represent a high-risk vulnerability. Two warnings may relate to file permissions or logging practices.
[action_custom_output] Sanitized 3 findings
[17:27:05.063] [Act] (_amphibious_automa.py:873) _PromptWorker:
{
    "content": "Analyzed the /var/app directory contents and system status. Key findings include exposed secrets in secrets.env (API_KEY and DB_PASS in plain

Notice the key difference from the previous examples:

- The worker uses **`output_schema=AuditReport`** — this tells the LLM to produce a typed `AuditReport` instance instead of making tool calls.
- Because the output is structured, the framework routes to **`action_custom_output()`** instead of `action_tool_call()`.
- Inside the hook, `decision_result` is the `AuditReport` instance — you can inspect and modify its typed fields directly.

This hook is only relevant when using `output_schema`. For normal tool-call workflows, use `before_action` (pre-execution) or `after_action` (post-execution) instead.

### `after_action` — Updating Context After Execution

The `after_action()` hook runs after tool execution completes but before the result is returned. This is the ideal place to update custom context fields based on tool results — for example, tracking which documents have been analyzed or accumulating findings across steps.

Like `before_action`, it follows the **worker → agent delegation pattern**: the worker's `after_action()` runs first; if it is not overwrited, the agent-level `after_action()` is called.

In [ ]:
from pydantic import ConfigDict
from bridgic.amphibious._type import ActionResult


class AuditContext(CognitiveContext):
    """Custom context that tracks audit findings across steps."""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    files_checked: list = Field(default_factory=list, description="Directories that have been listed")
    findings_count: int = Field(default=0, description="Total number of tool actions completed")


class ContextUpdatingAgent(AmphibiousAutoma[AuditContext]):
    auditor = think_unit(
        CognitiveWorker.inline(
            "Audit the system and report security findings. "
        ),
        max_attempts=3,
    )

    async def after_action(self, step_result, ctx: AuditContext):
        """Update custom context fields based on tool results."""
        action_result = step_result.result
        if not isinstance(action_result, ActionResult):
            return

        for step in action_result.results:
            if not step.success:
                continue
            ctx.findings_count += 1
            if step.tool_name == "list_files":
                ctx.files_checked.append(step.tool_arguments.get("directory", ""))

        print(f"[after_action] Files checked: {ctx.files_checked}, "
              f"Total actions: {ctx.findings_count}")

    async def on_agent(self, ctx: AuditContext):
        await self.auditor


agent = ContextUpdatingAgent(llm=llm, verbose=True)
result = await agent.arun(
    goal="Audit the /var/app directory for security issues",
    tools=[list_files_tool, read_file_tool, check_security_status_tool],
)
print(f"\nFinal context — Files checked: {agent.context.files_checked}, "
      f"Total actions: {agent.context.findings_count}")

[17:31:23.553] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[17:31:23.554] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[17:31:23.554] [Observe] (_amphibious_automa.py:861) _PromptWorker: None
[17:31:31.053] [Think] (_amphibious_automa.py:867) _PromptWorker: finish=False, step=Starting the security audit of /var/app directory. First step is to list all files and subdirectories to identify potential targets for further inspection.
[after_action] Files checked: ['/var/app'], Total actions: 1
[17:31:31.056] [Act] (_amphibious_automa.py:873) _PromptWorker:
{
    "content": "Starting the security audit of /var/app directory. First step is to list all files and subdirectories to identify potential targets for further inspection.",
    "result": {
        "results": [
            {
                "tool_id": "call_0",
                "tool_name": "list_files",
                "tool_arguments": {
                    "directory": "/var/app"
                },

## Hook Summary

Here is the complete reference for all OTC hooks:

| Hook | Override At | Default Behavior | Use Case |
|------|-----------|-----------------|----------|
| `observation()` | Worker / Agent | Worker → `_DELEGATE`; Agent → `None` | Inject environment perception |
| `thinking()` | Worker | Must implement | Define the thinking prompt |
| `build_messages()` | Worker | Standard assembly | Customize LLM message format |
| `before_action()` | Worker / Agent | Worker → `_DELEGATE`; Agent → passthrough | Intercept/filter tool calls |
| `action_tool_call()` | Agent | Concurrent execution | Custom tool execution strategy |
| `action_custom_output()` | Agent | Passthrough | Post-process `output_schema` results |
| `after_action()` | Worker / Agent | Worker → `_DELEGATE`; Agent → no-op | Update context after execution |

> **Note:** `action_tool_call()` and `action_custom_output()` are mutually exclusive paths — the framework routes to one or the other based on whether the worker uses `output_schema`.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored the full hook system of the OTC cycle:

- **Observe hooks** (`observation()`) let you inject custom environment perception at both the Worker and Agent level. Workers can delegate to the agent via `_DELEGATE`.
- **Think hooks** (`build_messages()`) let you reshape the messages sent to the LLM — useful for injecting safety rules or custom instructions.
- **Act hooks** (`before_action()`, `action_tool_call()`, `action_custom_output()`, `after_action()`) let you intercept tool calls before execution, customize how tools run, post-process outputs, and update custom context fields after execution.
- The **delegation pattern** (`_DELEGATE`) provides a clean two-tier hook system: per-worker customization with agent-level fallback.

These hooks give you fine-grained control over agent behavior without modifying the core framework logic.